## Example 3 - Long Short-Term Memory (LSTM)

Copyright 2024, Joshua Starmer, 편집 및 수정: YoungDon Choi (K-water Research Institute)

### 노트북 개요

> **📌 이 Notebook에서 다룰 내용은**
>
>  **PyTorch + Lightning**을 이용해 StatQuest의 "Long Short-Term Memory, Clearly Explained!!!" 영상에서 다룬 **LSTM 유닛**을 직접 만들고, 학습시키고, 예측까지 해볼 것입니다.
>
> - 학습 데이터는 **회사 A**와 **회사 B**의 4일치 주가(day1~day4)이며, 목표는 **5일째(day5) 값**을 예측하는 것입니다.
> - 자세히 보면 두 회사 데이터는 **1일째와 5일째**에서만 차이가 납니다. 즉 LSTM은 5일째를 예측하려면 아주 초반(1일째)에 일어난 일을 계속 "기억"하고 있어야 합니다. 이것이 바로 LSTM이 왜 필요한지를 보여주는 핵심 포인트입니다.
>
> 이 튜토리얼에서 할 4가지 작업은 다음과 같습니다.
> 1. **[Lightning으로 LSTM 유닛을 직접(손으로) 만들기](#build)**
> 2. **[LSTM을 학습시키고 Lightning·TensorBoard로 평가하기](#train)**
> 3. **[처음부터 다시 하지 않고 추가로 에폭 학습하기](#add_epochs)**
> 4. **[`nn.LSTM()`으로 LSTM을 만들고 Lightning으로 학습하기](#nnLSTM)**

<img src="./images/lstm_image.001.png" alt="A Long Short-Term Memory Unit" style="width: 1620px;">


<img src="./images/company_data.png" alt="Data for Companies A and B" style="width: 800px;">

### 신경망을 만들고 학습시키는 데 필요한 파이썬 모듈(module, 라이브러리)들을 불러오는(import) 단계

> `lightning` 패키지가 설치되어 있는지 확인하고, 없으면 자동으로 설치하는 코드입니다.
> - `%%capture`는 이 셀이 화면에 잔뜩 출력하는 표준에러(STDERR) 메시지를 숨겨주는 Jupyter "매직 명령어"입니다.
> - `try: __import__("lightning") except ImportError: pip.main(['install', "lightning"])` — 먼저 `lightning`을 불러와보고(import), 실패하면(설치가 안 되어 있으면) `pip`으로 설치합니다.
> - 만약 새로 설치가 이루어졌다면, 파이썬이 새 모듈을 인식하도록 세션(런타임)을 재시작해야 할 수 있다는 안내가 주석에 담겨 있습니다.

> 이번 노트북에서 사용할 나머지 라이브러리를 불러옵니다.
> - `torch`, `torch.nn`, `Adam` — 텐서 생성, 신경망 구성 요소, 최적화 알고리즘입니다.
> - `torch.nn.functional`(주석 처리됨) — 이번 노트북에서는 활성화 함수(시그모이드, 하이퍼볼릭탄젠트)를 `F.xxx()` 형태가 아니라 `torch.sigmoid()`, `torch.tanh()`처럼 직접 호출하므로 굳이 불러올 필요가 없어서 주석 처리되어 있습니다.
> - `lightning as L`, `TensorDataset`, `DataLoader` — 학습 코드를 간결하게 만들고 데이터를 배치로 다루기 위한 도구입니다.
>
> 실행 중 오류가 나면 패키지 설치 후 세션을 재시작하지 않아서인 경우가 많다는 안내가 이어집니다.

In [ ]:
import torch # torch will allow us to create tensors.
import torch.nn as nn # torch.nn allows us to create a neural network.
# import torch.nn.functional as F # nn.functional give us access to the activation and loss functions.
from torch.optim import Adam # optim contains many optimizers. This time we're using Adam

import lightning as L # lightning has tons of cool tools that make neural networks easier
from torch.utils.data import TensorDataset, DataLoader # these are needed for the training data

### PyTorch + Lightning으로 **LSTM 유닛을 직접(손으로) 구현**

LSTM도 신경망의 한 종류이므로, 지금까지와 마찬가지로 `LightningModule`을 상속받는 새 클래스를 만들며, 다음 메서드들을 정의합니다.
>
> - `__init__()` — 가중치·편향을 초기화하고 기타 준비를 합니다.
> - `lstm_unit()` — LSTM의 핵심 수식 계산을 담당합니다. 예를 들어 장기 기억(long-term memory)을 얼마나 유지할지(퍼센트)를 계산하는 부분입니다.
> - `forward()` — LSTM을 "펼쳐서(unroll)" 순전파를 수행합니다. 즉 학습 데이터의 각 시점(날짜)마다 `lstm_unit()`을 반복 호출합니다.
> - `configure_optimizers()` — 최적화 알고리즘을 설정합니다. 이전 장들에서는 `SGD`를 썼지만, 이번에는 **`Adam`**을 사용합니다.
> - `training_step()` — 학습 데이터를 `forward()`에 넣어 손실(loss)을 계산하고, 로그 파일에 기록합니다.

> LSTM 유닛을 직접 구현한 `LSTMbyHand` 클래스의 핵심 코드입니다. 다소 길지만 하나씩 뜯어보면 다음과 같습니다.
>
> **`__init__()` — 가중치·편향 초기화**
> - `L.seed_everything(seed=42)`로 난수 시드를 고정해서, 매번 같은 무작위 초기값에서 시작하도록 합니다.
> - `torch.normal(mean=0.0, std=1.0)`로 표준정규분포에서 뽑은 값으로 가중치들을 초기화하고, 모든 편향은 0으로 시작합니다(`nn.LSTM()`은 원래 균등분포를 쓰지만, 여기서는 정규분포로 초기화하고 있고, 균등분포 버전 코드는 주석으로 남겨둠).
> - LSTM은 크게 **3단계**로 구성되고, 각 단계마다 별도의 가중치·편향 세트를 가집니다.
>   1. **1단계 (Forget Gate, 망각 게이트)**: `wlr1, wlr2, blr1` — 현재의 **장기 기억(long-term memory)** 중 몇 %를 계속 기억할지 결정합니다.
>   2. **2단계 (Input Gate, 입력 게이트)**: `wpr1, wpr2, bpr1`(새로운 후보 기억을 얼마나 반영할지 비율) + `wp1, wp2, bp1`(새로운 후보 장기 기억 자체) — 새로운 정보를 얼마나, 어떻게 장기 기억에 추가할지 결정합니다.
>   3. **3단계 (Output Gate, 출력 게이트)**: `wo1, wo2, bo1` — 갱신된 장기 기억을 바탕으로 새로운 **단기 기억(short-term memory)** 을 얼마나 출력할지 결정합니다.
>
> **`lstm_unit(self, input_value, long_memory, short_memory)` — 한 시점(하루)에서의 LSTM 계산**
> - `long_remember_percent = sigmoid(short_memory*wlr1 + input_value*wlr2 + blr1)` — **망각 게이트**: 시그모이드 함수로 0~1 사이 값을 만들어, 기존 장기 기억을 몇 % 유지할지 계산합니다.
> - `potential_remember_percent = sigmoid(...)`와 `potential_memory = tanh(...)` — **입력 게이트**: 시그모이드로 "새 정보를 얼마나 반영할지 비율"을, 하이퍼볼릭탄젠트(tanh)로 "새로 들어올 후보 기억값(−1~1)"을 각각 계산합니다.
> - `updated_long_memory = long_memory*long_remember_percent + potential_remember_percent*potential_memory` — 기존 장기 기억 중 유지할 부분과, 새로 추가할 부분을 더해서 **장기 기억을 갱신**합니다.
> - `output_percent = sigmoid(...)` — **출력 게이트**: 갱신된 장기 기억 중 몇 %를 단기 기억(출력)으로 내보낼지 계산합니다.
> - `updated_short_memory = tanh(updated_long_memory) * output_percent` — 갱신된 장기 기억을 tanh로 -1~1 범위로 눌러준 뒤, 출력 비율을 곱해 **새로운 단기 기억**을 만듭니다.
> - 갱신된 `[updated_long_memory, updated_short_memory]`를 반환합니다.
>
> **`forward(self, input)` — LSTM을 4일치 데이터에 대해 펼쳐서(unroll) 실행**
> - `long_memory`, `short_memory`를 0으로 시작합니다(장기 기억은 "셀 상태(cell state)", 단기 기억은 "은닉 상태(hidden state)"라고도 부릅니다).
> - `input[0]`~`input[3]`(1~4일째 값)을 하나씩 꺼내, `lstm_unit()`을 4번 반복 호출하면서 매번 `long_memory`, `short_memory`를 갱신합니다.
> - 마지막에 남은 `short_memory`가 바로 LSTM의 최종 출력(5일째 예측값)이 되어 반환됩니다.
>
> **`configure_optimizers()`**
> - `Adam(self.parameters())`을 반환합니다(기본 학습률 0.001 사용). 주석에서는 `lr=0.1`을 쓰면 훨씬 빨리 학습되지만, 기본값(0.001)을 써야 StatQuest 영상에서 사용한 것과 똑같은 최종 가중치·편향 값을 얻을 수 있다고 설명합니다.
>
> **`training_step(self, batch)`**
> - `input_i, label_i = batch`로 배치에서 입력(4일치 값)과 정답(5일째 실제 값)을 분리합니다.
> - `output_i = self.forward(input_i[0])`로 예측값을 계산합니다.
> - `loss = (output_i - label_i)**2`로 **오차 제곱(squared residual)** 을 손실로 사용합니다.
> - `self.log("train_loss", loss)`로 손실을 로그에 기록합니다.
> - 정답이 0이면(회사 A) `out_0`에, 1이면(회사 B) `out_1`에 예측값을 각각 기록합니다. 이렇게 회사별로 로그를 나눠 기록하면, 나중에 TensorBoard에서 회사별 학습 진행 상황을 따로 확인할 수 있습니다.

In [ ]:
## Here we are implementing an LSTM network by hand...
class LSTMbyHand(L.LightningModule):

    def __init__(self):

        super().__init__()

        L.seed_everything(seed=42)

        mean = torch.tensor(0.0)
        std = torch.tensor(1.0)

        self.wlr1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wlr2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.blr1 = nn.Parameter(torch.tensor(0.), requires_grad=True)

        self.wpr1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wpr2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.bpr1 = nn.Parameter(torch.tensor(0.), requires_grad=True)

        self.wp1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wp2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.bp1 = nn.Parameter(torch.tensor(0.), requires_grad=True)

        self.wo1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wo2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.bo1 = nn.Parameter(torch.tensor(0.), requires_grad=True)


    def lstm_unit(self, input_value, long_memory, short_memory):

        long_remember_percent = torch.sigmoid((short_memory * self.wlr1) +
                                              (input_value * self.wlr2) +
                                              self.blr1)

        potential_remember_percent = torch.sigmoid((short_memory * self.wpr1) +
                                                   (input_value * self.wpr2) +
                                                   self.bpr1)
        potential_memory = torch.tanh((short_memory * self.wp1) +
                                      (input_value * self.wp2) +
                                      self.bp1)

        updated_long_memory = ((long_memory * long_remember_percent) +
                       (potential_remember_percent * potential_memory))

        output_percent = torch.sigmoid((short_memory * self.wo1) +
                                       (input_value * self.wo2) +
                                       self.bo1)
        updated_short_memory = torch.tanh(updated_long_memory) * output_percent

        return([updated_long_memory, updated_short_memory])


    def forward(self, input):

        long_memory = 0 # long term memory is also called "cell state" and indexed with c0, c1, ..., cN
        short_memory = 0 # short term memory is also called "hidden state" and indexed with h0, h1, ..., cN
        day1 = input[0]
        day2 = input[1]
        day3 = input[2]
        day4 = input[3]

        long_memory, short_memory = self.lstm_unit(day1, long_memory, short_memory)

        long_memory, short_memory = self.lstm_unit(day2, long_memory, short_memory)

        long_memory, short_memory = self.lstm_unit(day3, long_memory, short_memory)

        long_memory, short_memory = self.lstm_unit(day4, long_memory, short_memory)

        return short_memory


    def configure_optimizers(self): # this configures the optimizer we want to use for backpropagation.

        return Adam(self.parameters())


    def training_step(self, batch): # take a step during gradient descent.
        input_i, label_i = batch # collect input
        output_i = self.forward(input_i[0]) # run input through the neural network
        loss = (output_i - label_i)**2 ## loss = squared residual

        self.log("train_loss", loss)

        if (label_i == 0):
            self.log("out_0", output_i)
        else:
            self.log("out_1", output_i)

        return loss

> `model = LSTMbyHand()`로 모델을 만들고, `model.named_parameters()`로 학습 전(무작위) 가중치·편향 값을 모두 출력합니다. 이어서 각 회사의 4일치 데이터(`[0., 0.5, 0.25, 1.]`은 회사 A, `[1., 0.5, 0.25, 1.]`은 회사 B — 첫째 날 값만 다름)를 모델에 넣어 예측값을 계산하고, 실제 관측값(회사 A=0, 회사 B=1)과 나란히 출력합니다.

In [ ]:
model = LSTMbyHand()

print("Before optimization, the parameters are...")
for name, param in model.named_parameters():
    print(name, param.data)

print("\nNow let's compare the observed and predicted values...")

print("Company A: Observed = 0, Predicted =",
      model(torch.tensor([0., 0.5, 0.25, 1.])).detach())
print("Company B: Observed = 1, Predicted =",
      model(torch.tensor([1., 0.5, 0.25, 1.])).detach())

### 손으로 만든 LSTM을 학습시키고, **Lightning**과 **TensorBoard**로 학습 과정을 평가: "Part 1 - 시작하기"

> **Lightning**으로 학습하면 손으로 만든 LSTM도 학습시키기가 매우 쉽다는 설명입니다. 해야 할 일은 학습 데이터를 만들어 `DataLoader`에 담는 것뿐입니다.

> 학습 데이터를 만드는 코드입니다.
> - `inputs = torch.tensor([[0., 0.5, 0.25, 1.], [1., 0.5, 0.25, 1.]])` — 첫 번째 행은 회사 A의 1~4일째 값, 두 번째 행은 회사 B의 1~4일째 값입니다.
> - `labels = torch.tensor([0., 1.])` — 각각의 5일째 실제(정답) 값입니다(회사 A는 0, 회사 B는 1).
> - `TensorDataset(inputs, labels)`로 입력·정답을 묶고, `DataLoader(dataset)`으로 감싸서 `dataloader`를 만듭니다.

In [ ]:
## create the training data for the neural network.
inputs = torch.tensor([[0., 0.5, 0.25, 1.], [1., 0.5, 0.25, 1.]])
labels = torch.tensor([0., 1.])

dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

> 이제 **Lightning Trainer**(`L.Trainer`)를 만들어 학습 데이터에 `fit()`할 것이라는 설명입니다. 우선 **2000 에폭**으로 시작하며, 이걸로 충분할 수도, 부족할 수도 있는데, 학습 후 예측값을 관측값과 비교해보면서 판단할 것입니다.

> `trainer = L.Trainer(max_epochs=2000)`으로 Trainer를 만들고(기본 학습률 0.001을 쓰는 Adam이라 학습이 느립니다), `trainer.fit(model, train_dataloaders=dataloader)`로 2000 에폭 동안 학습을 진행합니다.

In [ ]:
trainer = L.Trainer(max_epochs=2000) # with default learning rate, 0.001 (this tiny learning rate makes learning slow)
trainer.fit(model, train_dataloaders=dataloader)

> 2000 에폭 학습이 끝났으니, 예측이 얼마나 좋아졌는지 확인해볼 것이라는 설명입니다.

> 학습된 모델에 회사 A, 회사 B의 4일치 데이터를 각각 넣어 예측값을 계산하고, 실제 관측값과 함께 출력합니다.

In [ ]:
print("\nNow let's compare the observed and predicted values...")
print("Company A: Observed = 0, Predicted =", model(torch.tensor([0., 0.5, 0.25, 1.])).detach())
print("Company B: Observed = 1, Predicted =", model(torch.tensor([1., 0.5, 0.25, 1.])).detach())

> 안타깝게도 2000 에폭 후에도 예측이 여전히 형편없다는 결과입니다. 더 학습을 해야 할 것 같지만, 무작정 더 학습하기 전에 **더 학습하면 실제로 개선될지**를 미리 확인하는 것이 시간(그리고 비용) 낭비를 줄이는 데 도움이 됩니다.
>
> 이를 위해 학습 중 로그 파일에 기록해둔 손실값과 예측값을 **TensorBoard**로 시각화하는 방법을 설명합니다.
> - 아래 코드 셀을 실행하면 `%tensorboard` 매직 명령어가 TensorBoard 서버를 백그라운드에서 시작하고, 셀 출력 영역에 TensorBoard 화면을 바로 띄워줍니다.
> - **MyBinder**에서는 `localhost:6006`으로 접속할 수 없기 때문에 `jupyter-server-proxy`를 거쳐 `.../proxy/6006/` 주소로 연결합니다. 셀이 출력하는 링크를 클릭하면 TensorBoard를 새 브라우저 탭에서 크게 볼 수 있습니다.
> - 내 컴퓨터(로컬)에서 실행할 때는 셀 출력 화면을 보거나 `http://localhost:6006/`에 접속하면 됩니다.
>
> 이어서 아래 그래프들(`train_loss`, 회사 A 예측값 `out_0`, 회사 B 예측값 `out_1`)에 대한 해석이 나옵니다.
> - **손실(train_loss)**: 줄어들고 있지만 아직 갈 길이 멀어 보입니다.
> - **회사 A 예측(out_0)**: 처음에는 0에 가까워 괜찮았지만 학습 초반에 0.5까지 나빠졌다가, 다시 조금씩 작아지는 중입니다.
> - **회사 B 예측(out_1)**: 처음에는 0 근처로 매우 나빴지만 계속 좋아지고 있고, 더 학습하면 계속 개선될 여지가 있어 보입니다.
>
> 결론적으로 그래프들은 "더 학습하면 예측이 좋아질 것"을 시사하므로, 에폭을 더 추가하기로 합니다.
>
> **참고:** 그래프에 여러 개의 선이 뒤섞여 지저분해 보인다면, `lightning_logs` 디렉터리를 삭제하고 노트북을 처음부터 다시 실행하면 됩니다. 매번 학습할 때마다 새 로그 파일이 `lightning_logs`에 쌓이고 TensorBoard가 기본적으로 전부 그려버리기 때문입니다.

<img src="./images/train_loss_2000_epochs.png" alt="Loss" style="width: 300px;"> <img src="./images/out_0_2000_epochs.png" alt="out_0" style="width: 300px;"> <img src="./images/out_1_2000_epochs.png" alt="out_1" style="width: 300px;">

In [ ]:
import os

# MyBinder(JupyterHub)에서는 브라우저가 서버의 localhost:6006에 직접 접속할 수 없으므로,
# jupyter-server-proxy 경로(<사용자 URL>/proxy/6006/)를 통해 TensorBoard를 띄웁니다.
# Route TensorBoard through jupyter-server-proxy when running under JupyterHub (e.g. MyBinder).
prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX")
if prefix:
    os.environ["TENSORBOARD_PROXY_URL"] = prefix.rstrip("/") + "/proxy/%PORT%/"
    print("새 탭에서 열기:", prefix.rstrip("/") + "/proxy/6006/")

%load_ext tensorboard
%tensorboard --logdir ./lightning_logs/ --port 6006

### 이미 학습된 모델에 에폭을 이어서 더 학습시키는 방법: "Part 2 - 처음부터 다시 하지 않고 에폭 추가하기"

> **Lightning**을 쓰면 이전에 멈춘 지점부터 학습을 이어갈 수 있다는 장점을 설명합니다. Lightning이 학습 중 가중치·편향 변화를 기록하는 **체크포인트(checkpoint) 파일**을 자동으로 만들어두기 때문에, Trainer에게 체크포인트 위치만 알려주면 처음 2000 에폭을 다시 학습할 필요 없이 이어서 학습할 수 있습니다. 여기서는 **1000 에폭**을 추가로 더 학습합니다.

> - `path_to_checkpoint = trainer.checkpoint_callback.best_model_path` — 방금 학습에서 만들어진 (기본적으로 가장 최근) 체크포인트 파일 경로를 가져와 출력합니다.
> - `trainer = L.Trainer(max_epochs=3000)` — 새 Trainer를 만들되, 이전(2000)보다 1000 늘린 3000 에폭으로 설정합니다.
> - `trainer.fit(model, train_dataloaders=dataloader, ckpt_path=path_to_checkpoint)` — `ckpt_path`에 이전 체크포인트를 지정해서, 처음부터가 아니라 2000 에폭 지점부터 이어서 1000 에폭을 추가로 학습합니다.

In [ ]:
## First, find where the most recent checkpoint files are stored
path_to_checkpoint = trainer.checkpoint_callback.best_model_path ## By default, "best" = "most recent"
print("The new trainer will start where the last left off, and the check point data is here: " +
      path_to_checkpoint + "\n")

## Then create a new Lightning Trainer
trainer = L.Trainer(max_epochs=3000) # Before, max_epochs=2000, so, by setting it to 3000, we're adding 1000 more.
## And then call fit() using the path to the most recent checkpoint files
## so that we can pick up where we left off.
trainer.fit(model, train_dataloaders=dataloader, ckpt_path=path_to_checkpoint)

> 1000 에폭을 추가로 학습했으니, 예측값이 어떻게 바뀌었는지 확인해볼 것이라는 설명입니다.

> 회사 A, 회사 B의 데이터를 다시 모델에 넣어 예측값을 계산하고 관측값과 함께 출력합니다.

In [ ]:
print("\nNow let's compare the observed and predicted values...")
print("Company A: Observed = 0, Predicted =", model(torch.tensor([0., 0.5, 0.25, 1.])).detach())
print("Company B: Observed = 1, Predicted =", model(torch.tensor([1., 0.5, 0.25, 1.])).detach())

> 이전보다 예측이 훨씬 좋아졌다는 결과입니다. TensorBoard 페이지를 새로고침하면 갱신된 그래프를 볼 수 있습니다.
>
> 그래프에서 파란색 선은 추가로 학습한 1000 에폭 동안의 값을 나타냅니다. **손실**은 더 작아지고 있고, 두 회사에 대한 예측도 모두 개선되고 있습니다. 하지만 아직 더 좋아질 여지가 있어 보이므로, **2000 에폭을 추가로 더** 학습해보기로 합니다.

<img src="./images/train_loss_3000_epochs.png" alt="Loss" style="width: 300px;"> <img src="./images/out_0_3000_epochs.png" alt="out_0" style="width: 300px;"> <img src="./images/out_1_3000_epochs.png" alt="out_1" style="width: 300px;">

> 셀 23과 동일한 방식으로, 이번에는 체크포인트를 이용해 `max_epochs=5000`(이전 3000에서 2000 추가)으로 새 Trainer를 만들고 이어서 학습합니다.

In [ ]:
## First, find where the most recent checkpoint files are stored
path_to_checkpoint = trainer.checkpoint_callback.best_model_path ## By default, "best" = "most recent"
print("The new trainer will start where the last left off, and the check point data is here: " +
      path_to_checkpoint + "\n")

## Then create a new Lightning Trainer
trainer = L.Trainer(max_epochs=5000) # Before, max_epochs=3000, so, by setting it to 5000, we're adding 2000 more.
## And then call fit() using the path to the most recent checkpoint files
## so that we can pick up where we left off.
trainer.fit(model, train_dataloaders=dataloader, ckpt_path=path_to_checkpoint)

> 총 5000 에폭까지 학습을 마쳤으니, 이제 예측값을 다시 확인해볼 것이라는 설명입니다.

In [ ]:
print("\nNow let's compare the observed and predicted values...")
print("Company A: Observed = 0, Predicted =", model(torch.tensor([0., 0.5, 0.25, 1.])).detach())
print("Company B: Observed = 1, Predicted =", model(torch.tensor([1., 0.5, 0.25, 1.])).detach())

> 회사 A, 회사 B의 데이터를 모델에 넣어 예측값을 계산하고 관측값과 함께 출력합니다.

> 예측이 이제 꽤 좋아졌습니다. 회사 A의 예측값은 실제값 0에 매우 가깝고, 회사 B의 예측값은 실제값 1에 가깝습니다.
>
> TensorBoard 그래프를 새로고침해서 확인해보면, 진한 빨간색 선(추가된 2000 에폭 구간)에서 **손실**과 두 회사에 대한 예측값 모두 점점 평평해지는(수렴하는) 모습을 볼 수 있습니다. 이는 에폭을 더 추가해도 성능이 크게 개선되지 않을 것임을 시사하므로, 여기서 학습을 마칩니다.
>
> 마지막으로, 최종적으로 학습된 가중치·편향 값을 출력해볼 것이라는 설명입니다. 이 값들은 이론적으로 이 노트북 맨 위에 나온 LSTM 유닛 그림과 StatQuest의 "Long Short-Term Memory" 영상에서 사용한 값과 (반올림 오차 범위 내에서) 같아야 합니다.

<img src="./images/train_loss_5000_epochs.png" alt="Loss" style="width: 300px;"> <img src="./images/out_0_5000_epochs.png" alt="out_0" style="width: 300px;"> <img src="./images/out_1_5000_epochs.png" alt="out_1" style="width: 300px;">

> `model.named_parameters()`로 5000 에폭 학습이 끝난 후 최종 가중치·편향 값을 모두 출력합니다. 이 값들이 LSTM 유닛의 각 게이트(망각·입력·출력)를 구성하는 최적화된 파라미터입니다.

In [ ]:
print("After optimization, the parameters are...")
for name, param in model.named_parameters():
    print(name, param.data)

### PyTorch가 이미 제공하는 **`nn.LSTM()`** 함수를 활용하는 방법

> nn.LSTM()`을 쓰면 `__init__()`과 `forward()` 코드가 훨씬 간결해집니다. 또한 이번에는 StatQuest 영상에서 쓴 것과 똑같은 파라미터 값을 재현하려는 목적이 없으므로, Adam의 학습률을 **0.1**로 크게 설정해서 학습 속도를 훨씬 빠르게 만들 수 있습니다. 그 외 나머지 구조는 손으로 만든 버전과 동일합니다.

> `nn.LSTM()`을 활용한 `LightningLSTM` 클래스입니다. 손으로 만든 `LSTMbyHand`보다 훨씬 짧습니다.
>
> **`__init__()`**
> - `L.seed_everything(seed=42)`로 시드를 고정합니다.
> - `self.lstm = nn.LSTM(input_size=1, hidden_size=1)` — PyTorch가 제공하는 LSTM 레이어 하나를 만듭니다. `lstm_unit()`에서 직접 정의했던 12개의 가중치·편향(w1~wo1, b1~bo1)이 `nn.LSTM()` 내부에서 자동으로 생성·관리됩니다.
>   - `input_size=1` — 각 시점(날짜)마다 들어오는 입력 변수(feature)가 1개(주가 값 하나)라는 뜻입니다.
>   - `hidden_size=1` — 출력(은닉 상태)의 차원이 1이라는 뜻입니다. 예를 들어 `hidden_size=50`으로 설정하면 출력이 50차원이 되어, 이후 다른 완전연결층(fully connected layer)의 입력 50개로 쓸 수도 있습니다.
>
> **`forward(self, input)`**
> - `input_trans = input.view(len(input), 1)` — 입력 벡터를 `nn.LSTM()`이 기대하는 형태(시퀀스 길이 × 특성 개수)로 모양을 바꿉니다(transpose 개념).
> - `lstm_out, _ = self.lstm(input_trans)` — 4일치 데이터를 한 번에 LSTM에 통과시킵니다. `lstm_out`에는 매 시점(날짜)마다의 단기 기억(short-term memory) 값들이 모두 담겨 있습니다(손으로 만든 버전에서 4번 반복 호출했던 것과 동일한 계산을 `nn.LSTM()`이 내부적으로 한 번에 처리해줍니다).
> - `prediction = lstm_out[-1]` — 마지막 시점(4일째를 처리한 후)의 단기 기억 값을 최종 예측값(5일째 예측)으로 사용합니다.
>
> **`configure_optimizers()`**
> - `Adam(self.parameters(), lr=0.1)` — 학습률을 0.1로 크게 설정해서 학습 속도를 빠르게 합니다.
>
> **`training_step()`**
> - 손으로 만든 버전(`LSTMbyHand.training_step()`)과 완전히 동일한 로직입니다: 예측값과 정답의 오차 제곱을 손실로 계산하고, `train_loss`와 회사별(`out_0`/`out_1`) 예측값을 로그에 기록합니다.

> 이제 이 새 클래스로 모델을 만들어, 초기(무작위) 가중치·편향 값과 그 값으로 만든 예측을 출력해볼 것이라는 설명입니다.

In [ ]:
## Instead of coding an LSTM by hand, let's see what we can do with PyTorch's nn.LSTM()
class LightningLSTM(L.LightningModule):

    def __init__(self): # __init__() is the class constructor function, and we use it to initialize the Weights and Biases.

        super().__init__() # initialize an instance of the parent class, LightningModule.

        L.seed_everything(seed=42)

        self.lstm = nn.LSTM(input_size=1, hidden_size=1)


    def forward(self, input):
        ## transpose the input vector
        input_trans = input.view(len(input), 1)

        lstm_out, _ = self.lstm(input_trans)

        ## lstm_out has the short-term memories for all inputs. We make our prediction with the last one
        prediction = lstm_out[-1]
        return prediction


    def configure_optimizers(self): # this configures the optimizer we want to use for backpropagation.
        return Adam(self.parameters(), lr=0.1) ## we'll just go ahead and set the learning rate to 0.1


    def training_step(self, batch): # take a step during gradient descent.
        input_i, label_i = batch # collect input
        output_i = self.forward(input_i[0]) # run input through the neural network
        loss = (output_i - label_i)**2 ## loss = squared residual

        self.log("train_loss", loss)

        if (label_i == 0):
            self.log("out_0", output_i)
        else:
            self.log("out_1", output_i)

        return loss

> `model = LightningLSTM()`으로 모델을 만들고, 학습 전 파라미터 값을 출력합니다. 이어서 회사 A, 회사 B 데이터를 모델에 넣어 학습 전 예측값도 관측값과 함께 출력합니다.

> 예상대로 학습 전 예측은 좋지 않습니다. 이제 모델을 학습시킬 것인데, 학습률을 0.1로 높였기 때문에 **300 에폭**만으로도 충분하다는 설명입니다(손으로 만든 버전은 기본 학습률로 5000 에폭이 필요했던 것과 대조적입니다).

In [ ]:
model = LightningLSTM() # First, make model from the class

## print out the name and value for each parameter
print("Before optimization, the parameters are...")
for name, param in model.named_parameters():
    print(name, param.data)

print("\nNow let's compare the observed and predicted values...")
print("Company A: Observed = 0, Predicted =", model(torch.tensor([0., 0.5, 0.25, 1.])).detach())
print("Company B: Observed = 1, Predicted =", model(torch.tensor([1., 0.5, 0.25, 1.])).detach())

> - `trainer = L.Trainer(max_epochs=300, log_every_n_steps=2)` — 학습률을 높여 학습 속도가 빨라진 만큼, 300 에폭만 학습합니다. 에폭 수가 워낙 적기 때문에, 기본 로그 기록 주기(50 스텝마다)를 쓰면 그래프에 점이 거의 안 찍혀 보기 힘들어지므로, `log_every_n_steps=2`로 2스텝(학습 데이터가 2줄이므로 사실상 매 에폭)마다 로그를 기록하도록 설정합니다.
> - `trainer.fit(model, train_dataloaders=dataloader)` — 300 에폭 동안 학습합니다.
> - 이어서 학습이 끝난 후의 파라미터 값을 출력합니다.

In [ ]:
trainer = L.Trainer(max_epochs=300, log_every_n_steps=2)

trainer.fit(model, train_dataloaders=dataloader)

print("After optimization, the parameters are...")
for name, param in model.named_parameters():
    print(name, param.data)

Now that training is done, let's print out the new predictions...

> 회사 A, 회사 B 데이터를 학습된 모델에 넣어 예측값을 계산하고 관측값과 함께 출력합니다.

In [ ]:
print("\nNow let's compare the observed and predicted values...")
print("Company A: Observed = 0, Predicted =", model(torch.tensor([0., 0.5, 0.25, 1.])).detach())
print("Company B: Observed = 1, Predicted =", model(torch.tensor([1., 0.5, 0.25, 1.])).detach())

> 겨우 **300 에폭**만으로도 LSTM이 좋은 예측을 하고 있음을 확인합니다. 회사 A의 예측은 관측값 0에, 회사 B의 예측은 관측값 1에 각각 가깝습니다.
>
> TensorBoard 페이지를 새로고침해서 최신 그래프를 볼 수 있는데, 왼쪽의 "Runs" 목록에서 `version_0`, `version_1`, `version_2`는 체크 해제하고 `version_3`(방금 진행한 300 에폭 학습)만 선택하면, 이번 학습 결과만 따로 볼 수 있습니다.

> 세 그래프(`train_loss`, 회사 A 예측 `out_0`, 회사 B 예측 `out_1`) 모두 **500 스텝(=250 에폭)** 정도 이후부터 평평해지는(수렴하는) 모습을 보여줍니다. 이는 에폭을 더 늘려도 예측이 크게 개선되지 않을 것임을 뜻함

<img src="./images/selecting_run_version_3.png" alt="Loss" style="width: 300px;">

<img src="./images/train_loss_nn.lstm_300_epochs.png" alt="Loss" style="width: 300px;"><img src="./images/out_0_nn.lstm_300_epochs.png" alt="Loss" style="width: 300px;"><img src="./images/out_1_nn.lstm_300_epochs.png" alt="Loss" style="width: 300px;">

